In [35]:
!pip install rasterio geopandas folium matplotlib

In [36]:
import rasterio
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

In [37]:
!pip install earthengine-api geemap

In [38]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-lattytytyty')

In [39]:
thai = ee.FeatureCollection("FAO/GAUL/2015/level1")

chiangmai = thai.filter(ee.Filter.eq('ADM1_NAME','Chiang Mai'))

roi = chiangmai.geometry()

In [40]:
# Farm
suan_thongkham_farm = ee.Geometry.Point([98.844559,18.630035])
naph_phupha_farm = ee.Geometry.Point([98.450871,18.511640])
norlae_strawberry_farm = ee.Geometry.Point([99.05,19.30])
south_samoeng_zone = ee.Geometry.Point([98.70,18.45])

# Forest
mae_kampong_forest = ee.Geometry.Point([99.347722166,18.8801518218])
ban_wat_chan_forest = ee.Geometry.Point([98.3036831509,19.0744823363])
doi_lo_forest = ee.Geometry.Point([98.790537,18.454747])
mae_sa_forest = ee.Geometry.Point([98.90,18.91])

# Green area
doi_suthep_park = ee.Geometry.Point([98.91,18.81])
nong_buak_hat_park = ee.Geometry.Point([98.98,18.78])
rama9_garden = ee.Geometry.Point([98.97,18.82])
huay_kaew_park = ee.Geometry.Point([98.95,18.80])
nong_khiao_urban_forest = ee.Geometry.Point([98.99,18.81])

In [41]:
    ee.Authenticate()
    ee.Initialize(project='my-project')

In [42]:
import ee
import geemap

ee.Initialize(project='ee-lattytytyty')

Map = geemap.Map()

Map.addLayer(suan_thongkham_farm, {'color':'red'}, 'Farm')
Map.addLayer(naph_phupha_farm, {'color':'red'}, 'Farm2')

Map.addLayer(mae_kampong_forest, {'color':'green'}, 'Forest')
Map.addLayer(ban_wat_chan_forest, {'color':'green'}, 'Forest2')

Map.addLayer(doi_suthep_park, {'color':'blue'}, 'Green Area')
Map.addLayer(nong_buak_hat_park, {'color':'blue'}, 'Green Area2')

In [43]:
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(roi) \
    .filterDate('2020-01-01', '2020-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',20)) \
    .median()

In [44]:
ndvi = collection.normalizedDifference(['B8','B4']).rename('NDVI')

In [45]:
collection2024 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(roi) \
    .filterDate('2024-01-01', '2024-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',20)) \
    .median()

ndvi2024 = collection2024.normalizedDifference(['B8','B4'])

In [46]:
ndvi_change = ndvi2024.subtract(ndvi)
Map.addLayer(ndvi_change,
             {'min':-0.5,'max':0.5,'palette':['red','white','green']},
             'NDVI Change')

In [47]:
Map.addLayer(ndvi2024, ndvi_vis, 'NDVI 2024')
Map.addLayer(ndvi, ndvi_vis, 'NDVI 2020')

In [48]:
task = ee.batch.Export.image.toDrive(
    image=ndvi,
    description='NDVI2020',
    scale=10,
    region=roi
)
task.start()

In [49]:
Map.addLayer(roi, {}, 'Study Area')

In [50]:
Map = geemap.Map(center=[14.0,100.6], zoom=9)

ndvi_vis = {
    'min': -1,
    'max': 1,
    'palette': ['blue','white','green']
}

change_vis = {
    'min': -0.5,
    'max': 0.5,
    'palette': ['red','white','green']
}

Map.addLayer(ndvi, ndvi_vis, 'NDVI')
Map.addLayer(ndvi, ndvi_vis, 'NDVI 2020') # Changed ndvi2020 to ndvi
Map.addLayer(ndvi2024, ndvi_vis, 'NDVI 2024')
Map.addLayer(ndvi_change, change_vis, 'NDVI Change')
Map.addLayer(roi, {}, 'Study Area')
Map.centerObject(roi, 9)

Map

Map(center=[18.792236850995966, 98.73204090284445], controls=(WidgetControl(options=['position', 'transparent_…

In [51]:
import geemap.chart as chart

chart.image_histogram(
    ndvi,
    region=roi,
    scale=30,
    max_buckets=50,
    min_bucket_width=0.01,
    max_raw=100000,
    max_pixels=1e9
)

Figure(axes=[Axis(label='Reflectance (x1e4)', scale=LinearScale(), tick_format='0.0f'), Axis(label='Count', or…

# **สรุปผล**
จากกราฟ Histogram ของการเปลี่ยนแปลงค่า NDVI พบว่าค่าการเปลี่ยนแปลงส่วนใหญ่กระจุกตัวใกล้ศูนย์ แสดงให้เห็นว่าพื้นที่ส่วนใหญ่ไม่มีการเปลี่ยนแปลงของพืชพรรณอย่างมาก อย่างไรก็ตามมีบางพื้นที่ที่มีค่า NDVI เพิ่มขึ้น ซึ่งอาจแสดงถึงการเพิ่มขึ้นของพื้นที่สีเขียวหรือการฟื้นตัวของพืชพรรณ ขณะที่บางพื้นที่มีค่า NDVI ลดลง ซึ่งอาจสะท้อนถึงการลดลงของพืชพรรณหรือการเปลี่ยนแปลงการใช้ที่ดิน

จากแผนที่การเปลี่ยนแปลง NDVI พื้นที่ที่แสดงสีเขียวเป็นบริเวณที่ค่า NDVI เพิ่มขึ้น ส่วนพื้นที่ที่แสดงสีแดงเป็นบริเวณที่ค่า NDVI ลดลง ซึ่งอาจเกิดจากการเปลี่ยนแปลงของการใช้ที่ดิน เช่น การขยายตัวของพื้นที่เมือง การทำเกษตรกรรม หรือการตัดไม้ทำลายป่า

นอกจากนี้ การเปลี่ยนแปลงของค่า NDVI ยังอาจได้รับอิทธิพลจากปัจจัยด้านสิ่งแวดล้อม เช่น ความแตกต่างของฤดูกาล สภาพภูมิอากาศ ปริมาณน้ำฝน และสภาพความชื้นของดิน ซึ่งล้วนส่งผลต่อการเจริญเติบโตของพืชพรรณในพื้นที่

# **คำถามท้าย Lab**

1. ทำไมการวิเคราะห์ NDVI จากภาพถ่ายดาวเทียมหลายช่วงเวลาจึงมีความสำคัญ?

**ตอบ **เพื่อช่วยให้สามารถติดตามการเปลี่ยนแปลงของพืชพรรณในพื้นที่ได้อย่างต่อเนื่อง เช่น การเพิ่มขึ้นหรือลดลงของพื้นที่ป่า การเปลี่ยนแปลงพื้นที่เกษตรกรรม หรือผลกระทบจากภัยธรรมชาติ การใช้ข้อมูลหลายช่วงเวลาช่วยให้สามารถเปรียบเทียบสภาพพื้นที่ในอดีตและปัจจุบัน เพื่อประเมินแนวโน้มของการเปลี่ยนแปลงสิ่งแวดล้อมและการใช้ที่ดินได้

2. พื้นที่ที่มีการเปลี่ยนแปลง NDVI อย่างมีนัยสำคัญควรได้รับการตรวจสอบเพิ่มเติมอย่างไร?

**ตอบ** โดยการเปรียบเทียบกับภาพถ่ายดาวเทียมความละเอียดสูง จากข้อมูลการใช้ที่ดิน หรือการสำรวจภาคสนาม (field survey) เพื่อยืนยันสาเหตุของการเปลี่ยนแปลงและยังสามารถใช้ข้อมูลดาวเทียมจากช่วงเวลาอื่น ๆมาวิเคราะห์แนวโน้มของการเปลี่ยนแปลงในระยะยาวได้

3. ปัจจัยใดที่อาจทำให้ค่า NDVI เปลี่ยนแปลงโดยไม่เกี่ยวข้องกับการเปลี่ยนแปลงของพืชพรรณ?

**ตอบ** ค่า NDVI อาจเปลี่ยนแปลงได้จากหลายปัจจัยที่ไม่ได้เกี่ยวข้องกับการเปลี่ยนแปลงของพืชพรรณโดยตรง เช่น สภาพบรรยากาศ (เมฆ หมอก หรือฝุ่นละออง) ความแตกต่างของฤดูกาล มุมการรับแสงของดาวเทียม ความชื้นของดิน หรือเงาของภูมิประเทศ ปัจจัยเหล่านี้อาจส่งผลต่อค่าการสะท้อนของแสงในช่วงคลื่นต่าง ๆ ทำให้ค่า NDVI เปลี่ยนแปลงได้

4. หากต้องการเพิ่มความแม่นยำของการวิเคราะห์การเปลี่ยนแปลง NDVI ควรใช้เทคนิคอะไรเพิ่มเติม?

**ตอบ** ใช้เทคนิคเพิ่มเติม เช่น การทำ atmospheric correction การใช้ภาพหลายช่วงเวลา (time-series analysis) การใช้ข้อมูลจากดาวเทียมหลายแหล่ง หรือการใช้ดัชนีพืชพรรณอื่น ๆ เช่น EVI ร่วมกับ NDVI รวมไปถึงการใช้ข้อมูลภาคสนามเพื่อยืนยันผลการวิเคราะห์ยังช่วยเพิ่มความน่าเชื่อถือของผลลัพธ์

5. ทำไมการใช้ DEM จึงช่วยให้เข้าใจบริบทของ NDVI ได้ดีขึ้น?

**ตอบ** ช่วยให้เข้าใจลักษณะภูมิประเทศของพื้นที่ เช่น ความสูง ความลาดชัน และทิศทางของภูเขา ซึ่งมีผลต่อการกระจายตัวของพืชพรรณและค่า NDVI ตัวอย่างเช่น พื้นที่ที่มีระดับความสูงมากอาจมีชนิดพืชที่แตกต่างจากพื้นที่ราบ การใช้ DEM ร่วมกับ NDVI จึงช่วยให้สามารถตีความผลการวิเคราะห์ได้อย่างถูกต้องและเข้าใจบริบททางภูมิประเทศ